<!-- beginner-banner-v2 -->

> 🧭 <strong>비개발자 수강생 안내</strong> — 이 노트북에서 새로 배우는 것: 엑셀의 <strong>피벗 테이블·VLOOKUP</strong> 을 SQL 로 — <code>GROUP BY</code>, <code>JOIN</code>, CTE, 윈도우 함수.
>
> - 📖 강의 페이지: <a href="https://siapapa.github.io/courses/ai-sql-agent/day1/03-sql-advanced/" target="_blank" rel="noopener noreferrer">day1/03-sql-advanced</a>
> - 🆕 처음이라면 → <a href="https://siapapa.github.io/courses/ai-sql-agent/beginners-guide/" target="_blank" rel="noopener noreferrer">비개발자 학습 가이드</a>
> - 🔤 모르는 단어 → <a href="https://siapapa.github.io/courses/ai-sql-agent/appendix/glossary/" target="_blank" rel="noopener noreferrer">용어 사전</a>
> - 🛠️ 환경/접속 막힘 → <a href="https://siapapa.github.io/courses/ai-sql-agent/setup/" target="_blank" rel="noopener noreferrer">사전 준비</a> · <a href="https://siapapa.github.io/courses/ai-sql-agent/appendix/troubleshooting/" target="_blank" rel="noopener noreferrer">트러블슈팅</a>
>
> 외부 링크는 새 탭으로 열리도록 설정돼 있어 Colab 의 리디렉션 경고 페이지를 거치지 않습니다.<br/>
> <strong>셀은 위에서 아래로 차례대로 실행</strong>하세요. 시연용 코드(<code>구경만 하세요</code> 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 02. 집계 · 조인 · CTE · 윈도우 함수
> Day 1 · 3H · 소요 약 50분

## 학습 목표

- `GROUP BY` / `HAVING` 으로 집계 쿼리를 작성한다.
- `INNER` / `LEFT` / `SELF JOIN` 을 목적에 맞게 선택한다.
- 서브쿼리와 CTE의 가독성 차이를 이해한다.
- 윈도우 함수(`ROW_NUMBER`, `RANK`, `LAG`, `SUM OVER`)로 그룹 내 연산을 수행한다.

> **선행 조건:** 이 노트북은 `01_postgres_basics.ipynb` 에서 적재한 병원 데이터베이스를 그대로 사용합니다. 01 노트북을 먼저 실행해 주세요.

In [ ]:
%pip install -q psycopg2-binary sqlalchemy pandas tabulate

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다.
import os

def _load_secret(key: str, required: bool = True) -> None:
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

_load_secret("NEON_DSN", required=True)

print("Environment ready.")


In [ ]:
# Reuse the same Neon engine / run_query helper
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine(os.environ["NEON_DSN"])

# 주의: SQL 안에 LIKE '...%' 같이 `%` 가 들어가면 psycopg2 의 pyformat 파라미터 자리표시자와
# 충돌해 `TypeError: ... immutabledict is not a sequence` 가 발생합니다. text() 로 감싸 SQLAlchemy 가
# 자동 이스케이프하게 하고, engine 대신 connection 을 pandas 에 넘겨 빈 파라미터 전달 경로를 우회합니다.
def run_query(sql: str, title: str = ""):
    if title:
        print(f"\n[{title}]")
    print(f"SQL: {sql.strip()}\n")
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    print(df.to_string(index=False))
    print(f"({len(df)} rows)")
    return df

## 집계 함수와 `GROUP BY`

| 함수 | 설명 |
|---|---|
| `COUNT(*)` | 행 수 |
| `COUNT(col)` | NULL이 아닌 값 수 |
| `SUM(col)` | 합계 |
| `AVG(col)` | 평균 |
| `MAX(col)` / `MIN(col)` | 최대/최소 |

**흐름:** `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY`

`WHERE`는 그룹화 **전** 필터, `HAVING`은 그룹화 **후** 필터입니다.

In [ ]:
# 첫 번째 집계 쿼리 — JOIN + GROUP BY 의 기본 형태.
# 학습 포인트:
#   - `JOIN ... ON ...` : 두 테이블을 공통 컬럼으로 묶는다 (여기서는 department_id).
#   - `GROUP BY` : 같은 그룹의 행을 한 행으로 압축. SELECT 에 쓴 비집계 컬럼은 모두 GROUP BY 에 들어가야 한다.
#   - `COUNT(*)` : 그룹 내 행 수. 별칭 `doctor_count` 를 줘서 ORDER BY 에서 재사용.
run_query("""
    SELECT d.name AS department, COUNT(*) AS doctor_count
    FROM doctors doc
    JOIN departments d ON d.department_id = doc.department_id
    GROUP BY d.name
    ORDER BY doctor_count DESC
""", "진료과별 의사 수")

In [ ]:
# 월별 방문 통계
run_query("""
    SELECT
        TO_CHAR(visit_date, 'YYYY-MM') AS month,
        COUNT(*) AS visit_count,
        SUM(COALESCE(cost, 0)) AS total_cost,
        ROUND(AVG(cost), 0) AS avg_cost
    FROM visits
    WHERE status = 'completed'
    GROUP BY TO_CHAR(visit_date, 'YYYY-MM')
    ORDER BY month
""", "월별 방문 통계")

In [ ]:
# 비율(%) 계산 — 윈도우 함수의 첫 등장.
# 학습 포인트:
#   - `COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()` :
#       바깥쪽 SUM 이 모든 그룹의 COUNT 를 합한 "전체 합계" 가 된다 → 각 그룹 비율 계산.
#   - `OVER ()` 의 빈 괄호는 "전체 행에 대해 한 덩어리로" 라는 뜻.
#   - 100.0 (실수) 으로 나눠야 정수 나눗셈에서 0이 되는 함정을 피할 수 있다.
run_query("""
    SELECT
        blood_type,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS percentage
    FROM patients
    GROUP BY blood_type
    ORDER BY count DESC
""", "혈액형별 환자 분포 (%)")

In [ ]:
# HAVING: 3회 이상 방문한 환자
run_query("""
    SELECT
        p.name,
        COUNT(*) AS visit_count,
        SUM(COALESCE(v.cost, 0)) AS total_cost
    FROM visits v
    JOIN patients p ON p.patient_id = v.patient_id
    WHERE v.status = 'completed'
    GROUP BY p.name
    HAVING COUNT(*) >= 3
    ORDER BY visit_count DESC
""", "3회 이상 방문한 환자")

## JOIN — 테이블 결합

```
INNER JOIN   : A ∩ B  (양쪽 매칭만)
LEFT  JOIN   : A       (왼쪽 모두 + 매칭된 오른쪽)
RIGHT JOIN   :    B   (오른쪽 모두 + 매칭된 왼쪽)
FULL  JOIN   : A ∪ B  (양쪽 전부)
SELF  JOIN   : A ⋈ A  (같은 테이블을 두 번)
```

In [ ]:
# INNER JOIN — 양쪽 모두 매칭되는 행만
run_query("""
    SELECT
        p.name AS patient_name,
        d.name AS doctor_name,
        v.visit_date,
        v.chief_complaint
    FROM visits v
    INNER JOIN patients p ON p.patient_id = v.patient_id
    INNER JOIN doctors  d ON d.doctor_id = v.doctor_id
    WHERE v.status = 'completed'
    ORDER BY v.visit_date DESC
    LIMIT 10
""", "INNER JOIN — 완료된 진료 기록 (최근 10건)")

In [ ]:
# LEFT JOIN — 모든 환자의 방문 횟수 (0건 포함)
run_query("""
    SELECT
        p.name,
        COUNT(v.visit_id) AS visit_count
    FROM patients p
    LEFT JOIN visits v ON v.patient_id = p.patient_id
    GROUP BY p.patient_id, p.name
    ORDER BY visit_count, p.name
""", "LEFT JOIN — 모든 환자 방문 횟수")

In [ ]:
# LEFT JOIN + IS NULL — 한 번도 방문하지 않은 환자
run_query("""
    SELECT p.name, p.phone
    FROM patients p
    LEFT JOIN visits v ON v.patient_id = p.patient_id
    WHERE v.visit_id IS NULL
""", "LEFT JOIN + IS NULL — 미방문 환자")

In [ ]:
# SELF JOIN — 같은 진료과 의사 쌍
run_query("""
    SELECT
        a.name AS doctor_a,
        b.name AS doctor_b,
        d.name AS department
    FROM doctors a
    JOIN doctors b
      ON a.department_id = b.department_id
     AND a.doctor_id < b.doctor_id
    JOIN departments d ON d.department_id = a.department_id
    ORDER BY department, doctor_a
""", "SELF JOIN — 같은 진료과 의사 쌍")

## 서브쿼리 (Subquery)

서브쿼리는 "쿼리 안의 쿼리"입니다. 위치에 따라 의미가 달라집니다.

- **`WHERE`절**: 비교·필터용 (단일값 또는 `IN`)
- **`FROM`절**: 임시 테이블처럼 사용 (파생 테이블)
- **`EXISTS`**: 연관 행이 있는지만 확인 (성능 우위)

In [ ]:
# WHERE subquery — 평균 급여 이상 의사
run_query("""
    SELECT name, salary
    FROM doctors
    WHERE salary > (SELECT AVG(salary) FROM doctors)
    ORDER BY salary DESC
""", "평균 급여 이상 의사")

In [ ]:
# FROM subquery — 진료과별 최고 급여 의사
run_query("""
    SELECT sub.department, sub.doctor_name, sub.salary
    FROM (
        SELECT
            d.name AS department,
            doc.name AS doctor_name,
            doc.salary,
            ROW_NUMBER() OVER (PARTITION BY d.department_id ORDER BY doc.salary DESC) AS rn
        FROM doctors doc
        JOIN departments d ON d.department_id = doc.department_id
    ) sub
    WHERE sub.rn = 1
    ORDER BY sub.salary DESC
""", "진료과별 최고 급여 의사")

In [ ]:
# EXISTS subquery — 진단 기록이 있는 방문
run_query("""
    SELECT v.visit_id, p.name, v.visit_date
    FROM visits v
    JOIN patients p ON p.patient_id = v.patient_id
    WHERE EXISTS (
        SELECT 1 FROM diagnoses dg WHERE dg.visit_id = v.visit_id
    )
    ORDER BY v.visit_date DESC
    LIMIT 10
""", "EXISTS — 진단 기록이 있는 방문")

## CTE (Common Table Expressions)

`WITH ... AS (SELECT ...)` 구문으로 **복잡한 쿼리를 단계별로 이름을 붙여** 구성합니다. 중첩 서브쿼리보다 가독성·디버깅이 훨씬 좋아집니다.

In [ ]:
# CTE — 월별 방문 vs 전체 평균 비교
run_query("""
    WITH monthly_visits AS (
        SELECT
            TO_CHAR(visit_date, 'YYYY-MM') AS month,
            COUNT(*) AS cnt
        FROM visits
        WHERE status = 'completed'
        GROUP BY TO_CHAR(visit_date, 'YYYY-MM')
    ),
    avg_visits AS (
        SELECT AVG(cnt) AS avg_cnt FROM monthly_visits
    )
    SELECT
        mv.month,
        mv.cnt AS visits,
        ROUND(av.avg_cnt, 1) AS overall_avg,
        CASE WHEN mv.cnt > av.avg_cnt
             THEN '평균 이상'
             ELSE '평균 미만' END AS status
    FROM monthly_visits mv, avg_visits av
    ORDER BY mv.month
""", "CTE — 월별 방문 vs 평균")

## 윈도우 함수

`함수() OVER (PARTITION BY ... ORDER BY ...)`

`GROUP BY`와 달리 **행 수를 줄이지 않고** 각 행 옆에 계산값을 붙입니다. "그룹 안에서 순위/누적/이동평균"을 계산할 때 반드시 필요합니다.

In [ ]:
# 윈도우 함수 첫 정식 등장: ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)
# 의미:
#   - PARTITION BY department_id : 진료과별로 그룹을 나눈다 (GROUP BY 와 달리 행 수는 줄지 않음).
#   - ORDER BY salary DESC : 그룹 안에서 급여 높은 순으로 줄을 세운다.
#   - ROW_NUMBER() : 줄 선 순서를 1, 2, 3... 으로 매긴다.
# 결과: 각 의사가 (전체 순위 / 진료과 내 순위) 를 동시에 갖는다 — GROUP BY 로는 불가능한 패턴.
run_query("""
    SELECT
        name, salary, department_id,
        ROW_NUMBER() OVER (ORDER BY salary DESC) AS overall_rank,
        ROW_NUMBER() OVER (PARTITION BY department_id ORDER BY salary DESC) AS dept_rank
    FROM doctors
""", "ROW_NUMBER — 순위")

In [ ]:
# RANK vs DENSE_RANK — 동점 처리 차이
run_query("""
    SELECT
        name, salary,
        RANK()       OVER (ORDER BY salary DESC) AS rank_val,
        DENSE_RANK() OVER (ORDER BY salary DESC) AS dense_rank_val
    FROM doctors
    ORDER BY salary DESC
""", "RANK vs DENSE_RANK")

In [ ]:
# LAG — 전월 대비 증감
run_query("""
    SELECT
        TO_CHAR(visit_date, 'YYYY-MM') AS month,
        COUNT(*) AS visits,
        LAG(COUNT(*)) OVER (ORDER BY TO_CHAR(visit_date, 'YYYY-MM')) AS prev_month,
        COUNT(*) - LAG(COUNT(*)) OVER (ORDER BY TO_CHAR(visit_date, 'YYYY-MM')) AS diff
    FROM visits
    WHERE status = 'completed'
    GROUP BY TO_CHAR(visit_date, 'YYYY-MM')
    ORDER BY month
""", "LAG — 전월 대비 방문 증감")

In [ ]:
# SUM OVER — 누적 합계 + 3건 이동평균
run_query("""
    SELECT
        visit_date, cost,
        SUM(cost) OVER (ORDER BY visit_date) AS running_total,
        AVG(cost) OVER (ORDER BY visit_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_avg_3
    FROM visits
    WHERE status = 'completed' AND cost > 0
    ORDER BY visit_date
    LIMIT 15
""", "SUM OVER — 누적 진료비 + 이동평균")

## 실습 과제

**문제:** "각 의사별로 가장 최근에 진료한 환자 3명을 추출하세요."

힌트:
- CTE에서 `ROW_NUMBER()`로 의사별 최근 방문에 순번 부여
- 외부 쿼리에서 `rn <= 3` 필터
- 의사 이름·환자 이름·방문일 출력

추가 실습:
1. 3회 이상 방문한 환자의 **평균 진료비**를 구하세요.
2. `LEFT JOIN`으로 **방문이 한 건도 없는 의사**가 있는지 확인하세요.
3. `SUM() OVER`로 **의사별 진료비 누적 합계**를 계산하세요.

In [ ]:
# TODO: 각 의사별 가장 최근 환자 3명
# 여기에 구현하세요.

## 다음 노트북에서는…

SQL 기본기를 마쳤으니, 다음 노트북 **`03_schema_intelligence.ipynb`** 에서는 "LLM이 스키마를 어떻게 읽는가"를 다룹니다. 같은 병원 DB를 AI 친화적 스키마로 업그레이드하면서 `COMMENT ON`·FK·뷰의 효과를 직접 확인합니다.